In [0]:
# from pyspark.sql.functions import current_timestamp, lit
# from delta.tables import DeltaTable

# source_db = "hive_metastore.we_retail_lake"
# target_db = "hive_metastore.we_retail_hub"

# tables = spark.catalog.listTables(source_db)

# for t in tables:
#     table_name = t.name

#     if table_name == "dates":
#         print(f"Skipping: {table_name}")
#         continue

#     source_table = f"{source_db}.{table_name}"
#     target_table = f"{target_db}.{table_name}"

#     print(f"\nProcessing {source_table} -> {target_table}")

#     try:
#         # -----------------------------
#         # 1. Read source
#         # -----------------------------
#         df = spark.read.table(source_table)
#         df = df.drop('aud_load_timestamp')
#         # -----------------------------
#         # 2. Add audit column (source side)
#         # -----------------------------
#         df = df.withColumn("aud_load_timestamp", current_timestamp())
#         df = df.withColumn("aud_upd_timestamp",current_timestamp())
#         df = df.withColumn("active_flag", lit("Y"))
#         df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)

#         print(f"SUCCESS: {table_name}")

#     except Exception as e:
#         print(f"FAILED: {table_name} -> {e}")

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

source_db = "hive_metastore.we_retail_lake"
target_db = "hive_metastore.we_retail_hub"

table_config = {
    "dim_campaigns":    {"key": ["campaign_id", "campaign_name"]},
    "dim_customers":    {"key": ["first_name", "last_name", "email", "customer_id"]},
    "dim_products":     {"key": ["product_id", "product_name", "category", "brand", "origin_location"]},
    "dim_salespersons": {"key": ["salesperson_id", "salesperson_name", "salesperson_role"]},
    "dim_stores":       {"key": ["store_id", "store_name", "store_type", "store_location"]}
}

audit_cols = ["aud_load_timestamp", "aud_ld_files", "aud_upd_timestamp", "row_key", "row_hash", "active_flag"]

for tbl in spark.catalog.listTables(source_db):   

    table_name = tbl.name

    if table_name == "dates" or table_name not in table_config:
        continue

    print(f"\nProcessing {table_name}")

    try:
        # -----------------------------------------------------------
        # READ SOURCE & BUILD HASHES
        # -----------------------------------------------------------
        df = spark.read.table(f"{source_db}.{table_name}")

        if "aud_load_timestamp" in df.columns:
            df = df.drop("aud_load_timestamp")
        df = df.withColumn("aud_load_timestamp", current_timestamp())

        business_cols = [c for c in df.columns if c not in audit_cols]
        natural_keys  = table_config[table_name]["key"]

        df = (
            df
            .withColumn(
                "row_key",
                sha2(concat_ws("||", *[coalesce(col(c).cast("string"), lit("NULL")) for c in natural_keys]), 256)
            )
            .withColumn(
                "row_hash",
                sha2(concat_ws("||", *[coalesce(col(c).cast("string"), lit("NULL")) for c in business_cols]), 256)
            )
        )

        # -----------------------------------------------------------
        # BUILD t_row_hash ON TARGET IN-MEMORY (never written to disk)
        # -----------------------------------------------------------
        target            = DeltaTable.forName(spark, f"{target_db}.{table_name}")
        target_df         = target.toDF()
        target_biz_cols   = [c for c in business_cols if c in target_df.columns]

        target_with_hash = (
            target_df
            .withColumn(
                "t_row_hash",
                sha2(concat_ws("||", *[coalesce(col(c).cast("string"), lit("NULL")) for c in target_biz_cols]), 256)
            )
        ).alias("t")

        source = df.alias("s")

        # -----------------------------------------------------------
        # PRE-FLIGHT COUNTS
        # -----------------------------------------------------------
        rows_to_expire = (
            target_with_hash
            .join(source, col("t.row_key") == col("s.row_key"), "inner")
            .filter(col("t.active_flag") == "Y")
            .filter(col("t.t_row_hash") != col("s.row_hash"))
        ).count()

        active_target  = target_df.filter(col("active_flag") == "Y").alias("t")
        rows_to_insert = (
            source
            .join(active_target, col("s.row_key") == col("t.row_key"), "left_anti")
        ).count()

        # -----------------------------------------------------------
        # EARLY EXIT
        # -----------------------------------------------------------
        if rows_to_expire == 0 and rows_to_insert == 0:
            print(f"  NO CHANGES: {table_name} — source matches target, skipping.")
            continue

        print(f"  Rows to expire : {rows_to_expire}")
        print(f"  Rows to insert : {rows_to_insert}")

        # ===========================================================
        # STEP 1 — EXPIRE changed active records
        # ===========================================================
        if rows_to_expire > 0:
            (
                DeltaTable.forName(spark, f"{target_db}.{table_name}").alias("t")
                .merge(
                    target_with_hash
                        .join(source, col("t.row_key") == col("s.row_key"), "inner")
                        .filter(col("t.active_flag") == "Y")
                        .filter(col("t.t_row_hash") != col("s.row_hash"))
                        .select("s.*", col("t.row_key").alias("matched_row_key")),
                    "t.row_key = matched_row_key"
                )
                .whenMatchedUpdate(
                    set={
                        "active_flag":       lit("N"),
                        "aud_upd_timestamp": current_timestamp()
                    }
                )
                .execute()
            )

        # ===========================================================
        # STEP 2 — INSERT new + changed rows as active records
        # ===========================================================
        if rows_to_insert > 0:
            target_schema_cols = target.toDF().columns

            inserts = (
                source
                .join(active_target, col("s.row_key") == col("t.row_key"), "left_anti")
                .withColumn("active_flag",       lit("Y"))
                .withColumn("aud_upd_timestamp", current_timestamp())
                .drop("row_hash")
                .select([col(c) for c in target_schema_cols if c in df.columns])
            )

            inserts.write.format("delta") \
                .mode("append") \
                .saveAsTable(f"{target_db}.{table_name}")

        print(f"  SUCCESS: {table_name}")

    except Exception as err:                        
        print(f"  FAILED:  {table_name} -> {err}")

In [0]:
%sql
select tgt.row_key as tg_r_k, src.row_key as src_r_k from hive_metastore.we_retail_hub.dim_campaigns tgt  join hive_metastore.we_retail_lake.dim_campaigns src 
on tgt.row_key = src.row_key

In [0]:
%sql
select * from hive_metastore.we_retail_hub.dim_campaigns